# Generate Head Boundary Conditions for 2D transect ATS - Naches-0

Head extracted from Zhi's 3D ATS simulation for Naches

- file `global/cfs/cdirs/m1800/naches_run2_share/Naches-3`
- Information
    - "Time": from 11224 to 16059; unit is day;
        - 12600 = 190+365x34 --> 2014.07.09
        - 15969 = 274+365x43 --> 2023.10.01

Output of this script

- constant head at the starting and end points -> to drive the run0
- head at a typical year at the starting and end points -> to drive the run1
- transient head

**File History**

update 2026/1/29
- previously, it was a two-step extraction
    - 1. extract point water head on nersc
    - 2. process cyclic spinup and transient
- now, merge two steps into this notebook
    - merge with `10-Projects/2025-RCSFA-HillslopeFire/MaterialsData/OakCreek_from_sundar/ats_WTD_pressure_based_Aug1_2024.ipynb` on NERSC

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

update 2025/8/7
- correct site name to NF01
- a cleaned version putting all input data and notebooks together

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
# hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

In [ ]:
outputs={}

## extract point raw data from 3D ATS simulation

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd

import ats_xdmf as xdmf
import time
import random
import pandas
import os

from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import geopandas as gpd

### Config 3D ATS-flow results

In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
model_dir = '/global/cfs/cdirs/m1800/naches_run2_share/Naches-3'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

# Define raw output, and skip raw point data extraction if detect it
outputs['tmp_BChead_raw'] = f'../data-processed/{site_name}/tmp_bc_startend_raw.naches-3.h5'

# Check if file exists
if os.path.exists(outputs['tmp_BChead_raw']):
    flag_atsptextract = False
    print(f"File {outputs['tmp_BChead_raw']} exists, skipping extraction.")
else:
    flag_atsptextract = True
    print(f"File {outputs['tmp_BChead_raw']} not found, will extract data.")

### Prepare hillslope shape

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
#site_name = 'NF01'
#meshsize_nx = 100

m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
meshsize_nx = loaded_data['meshsize_nx'].flatten()[0]
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon - already a shapely object, no conversion needed
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
# In watershed-workflow 2.0, use the shapely Polygon directly
xsec_plg_dict_shply = xsec_plg

# convert to latlon crs using warp.shply() in v2.0
# reproj_xsec_plg = watershed_workflow.warp.shply(xsec_plg, crs_daymet, crs_latlon)

In [ ]:
hillslope_gdf

### Load surface/subsurface h5 files

In [ ]:
## a function to estimate WTD based on pressure

def get_ats_wtd_pressurebased(pressure_subsurface,visfile_surface, visfile_subsurface):
    # visfile_subsurface.centroids.shape -> (n_surface, 14, 3); 14 is soil layers, bottom-top
    iz_coord = visfile_subsurface.centroids[:,:,-1]
    
    ### Find Equivalent Surface and Subsurface ID based on cell centroids
    # take some care on the rounding of coordinates, can cause errors
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    # [remarks] prepare to perform a pairwise compare [n_surface,1,2] with [1,n_surface,2] -> returns [n_surface, n_surface]
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    assert surface_subsurface_IDs[:,1].shape == visfile_surface.centroids[:,1].shape, f"Shape mismatch: change the round precision in the surface/subsurface centroid coordinates"
    
    ### Estimate WTD based on pressure
    # pressure head
    ih = (pressure_subsurface - patm) / (rho * g) # dim=(time, n_surface, 14)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1) # dim=(time, n_surface)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    # WTD elevation
    # [remark] iH_rev=part1+part2; part1 is the relative distance between water table and the first saturated subsurface cell
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
    wtdep_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    pwdep_pressure_based = - np.minimum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    assert pressure_subsurface[:,:,1].shape == wtdep_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"
    
    ### Re-arrange WTD in accordance to surface IDs
    # As pressure obtained from subsurface does not follow surface ID, re-arrange this:
    head_rearranged  = head_pressure_based[:, subsurface_indices]
    wtdep_rearranged = wtdep_pressure_based[:, subsurface_indices]
    pwdep_rearranged = pwdep_pressure_based[:, subsurface_indices]
    
    return surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged

In [ ]:
if flag_atsptextract:
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    print(visfile_surface.times)

    tmp_year = 2026 # a standard year
    day_number0 = int(visfile_surface.times[0])
    date0 = datetime.strptime(f'{tmp_year}-{day_number0%365}', '%Y-%j')
    print(f"t0: {day_number0}={day_number0%365}+365x{day_number0//365}")
    print(f"t0: {day_number0%365} is {date0.strftime('%B %d')}")
    
    day_number1 = int(visfile_surface.times[-1])
    date1 = datetime.strptime(f'{tmp_year}-{day_number1%365}', '%Y-%j')
    print(f"t1: {day_number1}={day_number1%365}+365x{day_number1//365}")
    print(f"t1: {day_number1%365} is {date1.strftime('%B %d')}")
    
else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # subsurface
    start = time.time()
    visfile_subsurface = xdmf.VisFile(directory=model_dir,
                                      filename="ats_vis_data.h5", 
                                      mesh_filename="ats_vis_mesh.h5")
    visfile_subsurface.loadMesh(columnar=True)
    end = time.time()
    print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")
    
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    # subsurface pressure (time, xy-space and soil columns)
    start = time.time()
    pressure_subsurface = visfile_subsurface.getArray('pressure')
    end = time.time()
    print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")
    
    # get the WTD (based on surface ATS ID)
    start = time.time()
    surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                                    visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
    end = time.time()
    print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

else:
    print("Skipping: data already extracted")

### Extract water head at the start & end points of a 2D transect

In [ ]:
if flag_atsptextract:
    surface_x_coord = visfile_surface.centroids[:,0]
    surface_y_coord = visfile_surface.centroids[:,1]
    
    # print(surface_x_coord)
    # print(surface_x_coord.shape)
    # print(surface_y_coord)
    # print(surface_y_coord.shape)
    
    start_coords = (gdf_reloaded['lon'].iloc[0], gdf_reloaded['lat'].iloc[0])
    end_coords   = (gdf_reloaded['lon'].iloc[-1], gdf_reloaded['lat'].iloc[-1])
    
    # Get times from surface
    surface_times = visfile_surface.times
    print(f"Number of timesteps: {len(surface_times)}")
    print(surface_times)

else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # Compute Euclidean distances
    dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
    dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)
    
    # Get the index of the closest point
    start_index = np.argmin(dist_start)
    end_index = np.argmin(dist_end)
    
    # Get the minimum distances
    min_dist_start = dist_start[start_index]
    min_dist_end = dist_end[end_index]
    
    # Print results
    print(f"Index for start_coords: {start_index}, Minimum distance: {min_dist_start}")
    print(f"Index for end_coords: {end_index}, Minimum distance: {min_dist_end}")

else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # outputs['tmp_BChead_raw'] = f'../data-processed/{site_name}/tmp_bc_startend_raw.h5'
    # Extract data based on computed indices
    startpt_head_subsrf_vis = head_rearranged[:, start_index]
    endpt_head_subsrf_vis = head_rearranged[:, end_index]

    surface_ponded_depth = visfile_surface.getArray('surface-ponded_depth')
    startpt_head_srf_vis = surface_ponded_depth[:, start_index]
    endpt_head_srf_vis = surface_ponded_depth[:, end_index]

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(startpt_head_subsrf_vis, startpt_head_srf_vis, color='blue', alpha=0.5, s=10)
    ax.scatter(endpt_head_subsrf_vis, endpt_head_srf_vis, color='red', alpha=0.5, s=10)

    # Add 1:1 reference line
    min_val = min(startpt_head_subsrf_vis.min(),endpt_head_subsrf_vis.min())
    max_val = max(startpt_head_subsrf_vis.max(),endpt_head_subsrf_vis.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='1:1 line')
    
    ax.set_xlabel(f'Ponded water depth based on subsurface pressure')
    ax.set_ylabel('Other points ponded depth [m]')
    ax.set_title(f'Ponded water depth from surface vis file')
    #ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    
    # Write to HDF5
    with h5.File(outputs['tmp_BChead_raw'], "w") as hdf:
        hdf.create_dataset("Time", data=surface_times)
        hdf.create_dataset("startpt_head_subsrf_vis", data=startpt_head_subsrf_vis)
        hdf.create_dataset("endpt_head_subsrf_vis", data=endpt_head_subsrf_vis)
        hdf.create_dataset("startpt_head_srf_vis", data=startpt_head_srf_vis)
        hdf.create_dataset("endpt_head_srf_vis", data=endpt_head_srf_vis)
    
    print(f"Data successfully written to {outputs['tmp_BChead_raw']}")

else:
    print("Skipping: data already extracted")